# WalkThru — Gaussian Splatting (photoreal visual layer)

This trains a **3D Gaussian Splat** — the photoreal look (no melted textures, sharp from every angle). It reuses the camera solve from the MESH notebook, so run that FIRST (it checkpoints `images/` + `sparse/` to `MyDrive/WalkThru/runs/<RUN_NAME>/`).

**Two layers, one property:**
- This notebook → **splat** = what the buyer SEES (visual layer)
- The mesh notebook → **mesh** = what the character walks on (collision layer)
The viewer loads the splat for visuals and the mesh for physics. Splats have no surfaces, so they can NEVER provide collision on their own — that is why we always keep the mesh.

**Run this in a FRESH runtime** (Runtime ▸ Disconnect and delete runtime) — it uses pip/CUDA, NOT conda, so it must not share the mesh notebook's environment.
Runtime ▸ Change runtime type ▸ **T4 GPU** ▸ Save first.

In [ ]:
# CELL 1 — GPU + install gsplat (Colab-friendly Gaussian splatting trainer)
!nvidia-smi | head -6
!pip install -q gsplat torch torchvision 2>&1 | tail -3
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "No GPU — set Runtime ▸ Change runtime type ▸ T4 GPU"
print("✅ CELL 1 done")

In [ ]:
# CELL 2 — get the examples/simple_trainer (gsplat's turnkey COLMAP trainer) + its deps
import os
os.chdir("/content")
!git clone -q https://github.com/nerfstudio-project/gsplat.git
!pip install -q -r /content/gsplat/examples/requirements.txt 2>&1 | tail -3
print("trainer:", os.path.exists("/content/gsplat/examples/simple_trainer.py"))
print("✅ CELL 2 done")

In [ ]:
# CELL 3 — pull the checkpoint from the MESH notebook (images + sparse camera solve)
RUN_NAME = "myroom"   # <-- MUST match the mesh notebook's RUN_NAME
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
SRC = f"/content/drive/MyDrive/WalkThru/runs/{RUN_NAME}"
DATA = f"/content/data/{RUN_NAME}"
assert os.path.isdir(f"{SRC}/sparse/0"), f"No sparse solve at {SRC} — run the MESH notebook through CELL 6 first"
os.makedirs(f"{DATA}/sparse", exist_ok=True)
if not os.path.isdir(f"{DATA}/images"): shutil.copytree(f"{SRC}/images", f"{DATA}/images")
if not os.path.isdir(f"{DATA}/sparse/0"): shutil.copytree(f"{SRC}/sparse/0", f"{DATA}/sparse/0")
print("images:", len(os.listdir(f"{DATA}/images")), "| sparse:", os.listdir(f"{DATA}/sparse/0"))
print("✅ CELL 3 done")

In [ ]:
# CELL 4 — TRAIN the splat (~20-40 min on T4). Watch the loss fall; a preview .ply is written periodically.
import os
os.chdir("/content/gsplat/examples")
RESULT = f"/content/results/{RUN_NAME}"
# 7000 steps = fast/decent; 30000 = best. Start with 7000 to validate the capture.
!python simple_trainer.py default \
  --data-dir {DATA} \
  --data-factor 1 \
  --result-dir {RESULT} \
  --max-steps 7000 \
  --save-ply \
  --ply-steps 7000 \
  --disable-viewer 2>&1 | grep -Ei "step|loss|PSNR|ply|error|saved|Traceback" | tail -40
print("✅ CELL 4 done — send me the last few loss/PSNR lines")

In [ ]:
# CELL 5 — locate the trained .ply, copy to Drive, download
import glob, shutil, os
plys = sorted(glob.glob(f"{RESULT}/**/*.ply", recursive=True), key=os.path.getmtime)
assert plys, "No .ply produced — send CELL 4's full output"
src = plys[-1]
out = f"/content/{RUN_NAME}_splat.ply"
shutil.copy(src, out)
shutil.copy(src, f"/content/drive/MyDrive/WalkThru/runs/{RUN_NAME}/{RUN_NAME}_splat.ply")
print(subprocess.run(["bash","-lc",f"ls -lh {out}"], capture_output=True, text=True).stdout) if False else print(os.path.getsize(out)/1e6, "MB ->", out)
from google.colab import files
files.download(out)
print("✅ CELL 5 done — this .ply is your Gaussian splat (visual layer)")

## After downloading

You now have TWO files for the same property, sharing one camera solve so they line up perfectly:
- `<RUN_NAME>_splat.ply`  → photoreal **visual** layer (this notebook)
- `<RUN_NAME>_mesh.glb`   → **collision** layer (mesh notebook)

On the laptop, load both in the viewer:
```
http://localhost:5173/?splat=/scans/myroom_splat.ply&collision=/scans/myroom_mesh.glb
```
The viewer renders the splat and walks the character on the (invisible) mesh. If the two ever look mis-aligned, it's because the splat wasn't trained from the SAME sparse solve — re-run CELL 3 to be sure.

**Compress the splat** for the web with the SuperSplat / `.ksplat` toolchain later; raw training .ply is large.